In [1]:
# Importación e librerías

import pandas as pd
import plotly.express as px

# Análisis exploratorio de datos climatológicos para diseño bioclimático

In [2]:
# Importamos los datos
df = pd.read_csv('merida_clima.csv')
df.head()

,año,mes,dia,hora,minuto,indicador_datos,temp_bulbo_seco,temp_punto_rocio,humedad_relativa,presion_atmosferica,...,altura_techo_nubes,estado_presente,estado_pasado,agua_precipitable,profundidad_optica_aerosol,profundidad_nieve,dias_desde_ultima_nevada,albedo,precipitacion_liquida,cantidad_precipitacion
0,2022,1,1,1,0,?9?9?9?9E0?9?9?9?9?9?9?9?9?9?9?9*9?9*9*9?9*9,23.0,22.0,94,101190,...,77777,0,919999999,87,0.085,0,88,0.14,0.0,0.0
1,2022,1,1,2,0,?9?9?9?9E0?9?9?9?9?9?9?9?9?9?9?9*9?9*9*9?9*9,23.0,22.0,94,101190,...,77777,0,919999999,87,0.085,0,88,0.14,0.0,0.0
2,2022,1,1,3,0,?9?9?9?9E0?9?9?9?9?9?9?9?9?9?9?9*9?9*9*9?9*9,22.0,21.0,94,101189,...,77777,0,919999999,78,0.085,0,88,0.14,0.0,0.0
3,2022,1,1,4,0,?9?9?9?9E0?9?9?9?9?9?9?9?9?9?9?9*9?9*9*9?9*9,22.0,21.0,94,101189,...,498,0,919999999,78,0.085,0,88,0.14,0.0,0.0
4,2022,1,1,5,0,?9?9?9?9E0?9?9?9?9?9?9?9?9?9?9?9*9?9*9*9?9*9,23.0,22.0,94,101190,...,457,0,919999999,87,0.085,0,88,0.14,0.0,0.0


In [6]:
# descartamos existencia de datos ausentes
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 35 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   año                                  8760 non-null   int64  
 1   mes                                  8760 non-null   int64  
 2   dia                                  8760 non-null   int64  
 3   hora                                 8760 non-null   int64  
 4   minuto                               8760 non-null   int64  
 5   indicador_datos                      8760 non-null   str    
 6   temp_bulbo_seco                      8760 non-null   float64
 7   temp_punto_rocio                     8760 non-null   float64
 8   humedad_relativa                     8760 non-null   int64  
 9   presion_atmosferica                  8760 non-null   int64  
 10  radiacion_extraterrestre_horizontal  8760 non-null   int64  
 11  radiacion_extraterrestre_normal      8760

## Temperatura

Uno de los parámetros más importantes a considerar en el diseño bioclimatico es la temperatura, la cual nos dictaminará los requerimentos energéticos de nuestro proyecto. Para ello hay qeu visualizar el comportamiento de la temperatura a lo largo del año, para determinar los meses más caluros y más fríos.

In [ ]:
fig = px.histogram(df, x='temp_bulbo_seco', nbins=30,
                title='Distribución de Temperatura de Bulbo Seco — Mérida, Yuc.',
                labels={'temp_bulbo_seco': 'Temperatura (°C)'})
fig.show()

In [9]:
fig = px.scatter(df, x='temp_bulbo_seco', y='humedad_relativa',
                 color='mes',
                 title='Temperatura vs Humedad Relativa',
                 labels={'temp_bulbo_seco': 'Temperatura (°C)',
                         'humedad_relativa': 'Humedad Relativa (%)',
                         'mes': 'Mes'})
fig.show()

Los gráficos anteriores muestran en primera instancia las temperaturas dominantes, vemos que en general la media de temperatura se encuentra en los 25 °C. Por otra parte el segundo gráfico muestra la relación entre la temperatura y la humedad, lo cual nos da una correlación negativa, es decir, a mayor temperatura menor humedad. 

Veamos ahora los datos del promedio de temperatura por mes.

In [21]:
# datos promedio de temperatura por mes
df_temp_mes = (
    df
    .groupby('mes')
    .agg(temp_media=('temp_bulbo_seco', 'mean'))
    .reset_index()
    )

df_temp_mes['temp_media'] = round(df_temp_mes['temp_media'], 2)

In [22]:
df_temp_mes

,mes,temp_media
0,1,23.36
1,2,25.39
2,3,26.64
3,4,28.63
4,5,29.73
5,6,28.32
6,7,28.26
7,8,28.45
8,9,27.79
9,10,26.51


In [ ]:
# Gráfico de temp por mes

fig = px.line(df_temp_mes,
            x = 'mes',
            y = 'temp_media',
            markers = True,
            labels={
                    'temp_media': 'Temperatura (°C)',
                    'mes': 'Mes'
                },
                title='Temperatura media por mes')
fig.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Temperatura media por mes',
                                    'Distribución de temperatura por mes'))

# Línea
fig.add_trace(
    go.Scatter(x=df_temp_mes['mes'], y=df_temp_mes['temp_media'],
            mode='lines+markers', name='Temp. media'),
    row=1, col=1
)

# Caja — una por cada mes
for mes in sorted(df['mes'].unique()):
    fig.add_trace(
        go.Box(y=df[df['mes'] == mes]['temp_bulbo_seco'],
            name=str(mes)),
        row=1, col=2
    )

fig.update_xaxes(title_text='Mes', row=1, col=1)
fig.update_yaxes(title_text='Temperatura (°C)', row=1, col=1)
fig.update_xaxes(title_text='Mes', row=1, col=2)
fig.update_yaxes(title_text='Temperatura (°C)', row=1, col=2)

fig.show()